# Turn AC-COLAB-FULL-SHARDS-V1 -- resumable full-corpus Colab CUDA runner (v1)

Embeds ONE shard (of 8) of the full 441,879-row eligible KURE-v1 embedding
population with `nlpai-lab/KURE-v1` (revision
`4ed4540949c70b7da2c74004a915e1f2d5e46e4f`, dimension 1024, float32,
L2-normalized), using **1,000-row block checkpoints** so a Colab
disconnect loses at most one unfinished block, never the whole shard.

Reuses `gpu-benchmark-colab-cuda-runner-v2.ipynb`'s model loading,
preprocessing, NPY writer, and normalization code unchanged -- no new
model contract.

**Set `SHARD_ID` in Cell 1 to the shard you are running (0-7) before
executing.** Run this notebook once per shard (8 total executions across
however many Colab sessions you like -- each shard is independently
resumable).

**What this notebook reads (non-sensitive only):**
- `full-shard-export-manifest.json` (git-ignored, task-owned; shard
  boundaries/SHAs/model pins -- no chunk text)
- `kure-full-input-shard-NNN.jsonl.gz` for the selected `SHARD_ID`
  (gzip JSONL, 4 fields per row: `global_eligible_index`,
  `embedding_input_id`, `embed_text_sha256`, `text` -- competition-corpus
  chunk text ONLY, never Gold/expected_answer/DEV_CHECK/HOLDOUT/Owner
  decision/DB URL/API key)

**What it never does:** read/write Gold, DEV_CHECK, HOLDOUT, Owner
decisions, API keys, DB URLs, or the raw DocumentIR/PostgreSQL dump.
Never logs prompt/chunk text -- only ids, hashes, counts, and timings.
Never writes raw vectors as JSON -- only binary `.npy`.

**This Turn does NOT execute this notebook or upload/download anything on
the user's behalf** -- Claude Code has no Colab/GPU access. This is a
runnable template the user runs themselves, once per shard.

In [ ]:
"""Cell 1 -- environment, GPU, model, and shard selection. Fails closed
(raises, does not continue) if no GPU is present or the loaded model's
revision/dimension disagree with the pin."""
!pip install -q sentence-transformers==3.0.1

from google.colab import drive
drive.mount("/content/drive")

import json, time, hashlib, platform, struct, os
import numpy as np
import torch
import transformers
import sentence_transformers
from sentence_transformers import SentenceTransformer

# ---- set this before running ----
SHARD_ID = 0  # 0..7
# ----------------------------------

assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU required -- refusing to run on CPU"
GPU_NAME = torch.cuda.get_device_name(0)
CUDA_MEM_TOTAL_BYTES = torch.cuda.get_device_properties(0).total_memory

MODEL_REPOSITORY = "nlpai-lab/KURE-v1"
MODEL_REVISION = "4ed4540949c70b7da2c74004a915e1f2d5e46e4f"
DIMENSION = 1024
RUNNER = "COLAB_CUDA_FULL_SHARD"
DRIVE_DIR = "/content/drive/MyDrive/p11f0-full-shard-embedding"
BATCH_SIZE_INITIAL = 64  # halved on CUDA OOM, see Cell 5
BLOCK_SIZE = 1000  # rows per checkpointed block, section H

shard_str = f"{SHARD_ID:03d}"
SHARD_GZ_PATH = f"{DRIVE_DIR}/kure-full-input-shard-{shard_str}.jsonl.gz"
MANIFEST_PATH = f"{DRIVE_DIR}/full-shard-export-manifest.json"
SHARD_OUT_DIR = f"{DRIVE_DIR}/shard-{shard_str}"
os.makedirs(SHARD_OUT_DIR, exist_ok=True)

model = SentenceTransformer(MODEL_REPOSITORY, revision=MODEL_REVISION, device="cuda")
loaded_dim = model.get_sentence_embedding_dimension()
if loaded_dim != DIMENSION:
    raise RuntimeError(f"REVISION_OR_DIMENSION_MISMATCH: loaded model reports dimension={loaded_dim}, expected {DIMENSION} -- refusing to proceed")

runtime_versions = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "sentence_transformers": sentence_transformers.__version__,
}
print(json.dumps({"gpu": GPU_NAME, "shard_id": SHARD_ID, "runtime_versions": runtime_versions}, indent=2))

In [ ]:
"""Cell 2 -- sha256 / NPY helpers (hand-implemented, byte-for-byte matching
scripts/p11f0-colab-full-shard-verify.mjs's reader -- float32, C-order,
'<f4' descr, v1.0 header). Never writes raw vectors as JSON."""
import gzip

def sha256_hex(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def sha256_file(path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def write_npy_float32_matrix(path, arr: np.ndarray):
    assert arr.dtype == np.float32 and arr.ndim == 2 and arr.flags["C_CONTIGUOUS"]
    header = f"{{'descr': '<f4', 'fortran_order': False, 'shape': ({arr.shape[0]}, {arr.shape[1]}), }}"
    pre_header_len = 6 + 2 + 2
    unpadded = header + "\n"
    total = pre_header_len + len(unpadded)
    padding = (64 - (total % 64)) % 64
    header_bytes = (header + " " * padding + "\n").encode("ascii")
    with open(path, "wb") as f:
        f.write(b"\x93NUMPY")
        f.write(bytes([1, 0]))
        f.write(struct.pack("<H", len(header_bytes)))
        f.write(header_bytes)
        f.write(arr.tobytes(order="C"))

def read_npy_float32_matrix(path):
    import ast as _ast
    with open(path, "rb") as f:
        buf = f.read()
    assert buf[1:6] == b"NUMPY"
    header_len = struct.unpack("<H", buf[8:10])[0]
    header = buf[10:10 + header_len].decode("ascii")
    # ast.literal_eval, never eval() -- the header is a trusted Python dict
    # literal by the NPY format itself, but literal_eval refuses anything
    # beyond literals (no arbitrary code execution) regardless.
    shape = _ast.literal_eval(header)["shape"]
    data_start = 10 + header_len
    arr = np.frombuffer(buf[data_start:], dtype="<f4").reshape(shape)
    return arr

In [ ]:
"""Cell 3 -- load the shard manifest, decompress THIS shard's gzip input,
independently re-verify every row's embed_text_sha256 (never trust the
file's own claim), the global index sequence, and the manifest's own
membership/ordering SHA pins. Fails closed on any mismatch."""
with open(MANIFEST_PATH) as f:
    export_manifest = json.load(f)

if export_manifest["model"]["repository"] != MODEL_REPOSITORY or export_manifest["model"]["revision"] != MODEL_REVISION or export_manifest["model"]["dimension"] != DIMENSION:
    raise RuntimeError("MANIFEST_PIN_MISMATCH: full-shard-export-manifest.json model pins do not match this notebook's MODEL_REPOSITORY/MODEL_REVISION/DIMENSION")

shard_meta = next((s for s in export_manifest["shards"] if s["shard_index"] == SHARD_ID), None)
if shard_meta is None:
    raise RuntimeError(f"SHARD_NOT_IN_MANIFEST: shard_index={SHARD_ID} not found in full-shard-export-manifest.json")

gz_actual_sha256 = sha256_file(SHARD_GZ_PATH)
if gz_actual_sha256 != shard_meta["compressed_file_sha256"]:
    raise RuntimeError(f"SHARD_FILE_SHA_MISMATCH: manifest says {shard_meta['compressed_file_sha256']}, actual {gz_actual_sha256}")

with gzip.open(SHARD_GZ_PATH, "rt", encoding="utf-8") as f:
    raw_lines = [line for line in f if line.strip()]

rows = []
for line in raw_lines:
    row = json.loads(line)
    actual_text_sha = sha256_hex(row["text"])
    if actual_text_sha != row["embed_text_sha256"]:
        raise RuntimeError(f"TEXT_SHA_MISMATCH at global_eligible_index={row['global_eligible_index']}: file claims {row['embed_text_sha256']}, recomputed {actual_text_sha}")
    rows.append(row)

# Deterministic order: sort explicitly by global_eligible_index, never trust
# on-disk order alone -- then verify it is exactly the manifest's own
# contiguous [global_start_index, global_end_index] range with no gap/dup.
rows.sort(key=lambda r: r["global_eligible_index"])
expected_indices = list(range(shard_meta["global_start_index"], shard_meta["global_end_index"] + 1))
actual_indices = [r["global_eligible_index"] for r in rows]
if actual_indices != expected_indices:
    raise RuntimeError(f"GLOBAL_INDEX_RANGE_MISMATCH: shard {SHARD_ID} expected {expected_indices[0]}..{expected_indices[-1]} ({len(expected_indices)} rows), got {len(actual_indices)} rows starting {actual_indices[0] if actual_indices else None}")
if len(rows) != shard_meta["row_count"]:
    raise RuntimeError(f"ROW_COUNT_MISMATCH: expected {shard_meta['row_count']}, got {len(rows)}")

ids_in_order = [r["embedding_input_id"] for r in rows]
if len(set(ids_in_order)) != len(ids_in_order):
    raise RuntimeError("DUPLICATE_EMBEDDING_INPUT_ID_IN_SHARD")

print(json.dumps({"shard_id": SHARD_ID, "row_count": len(rows), "global_range": [expected_indices[0], expected_indices[-1]]}, indent=2))

In [ ]:
"""Cell 4 -- deterministic block plan + resume scan. Each block is a fixed,
deterministic, contiguous BLOCK_SIZE-row slice of this shard (never
row-skipping, never a different block boundary across runs). A block is
considered ALREADY DONE only if: its completion marker exists AND its
vectors/mapping/manifest files' own recorded SHA-256 match their actual
bytes AND row_count/dimension/dtype/global_range/model_revision in its
manifest match what THIS run expects. Any mismatch -- corrupt, partial,
or from a different model revision/config -- makes the block eligible for
a full re-run (never a partial/row-level patch)."""
def block_ranges(total_rows, block_size):
    ranges = []
    start = 0
    while start < total_rows:
        end = min(start + block_size, total_rows)
        ranges.append((start, end))  # shard-local [start, end)
        start = end
    return ranges

blocks = block_ranges(len(rows), BLOCK_SIZE)

def block_paths(block_idx):
    prefix = f"{SHARD_OUT_DIR}/block-{block_idx:04d}"
    return {
        "vectors": f"{prefix}-vectors.npy",
        "mapping": f"{prefix}-row-mapping.jsonl",
        "manifest": f"{prefix}-manifest.json",
        "marker": f"{prefix}-COMPLETE",
    }

def block_is_valid(block_idx, local_start, local_end):
    paths = block_paths(block_idx)
    if not os.path.exists(paths["marker"]):
        return False, "NO_COMPLETION_MARKER"
    for key in ("vectors", "mapping", "manifest"):
        if not os.path.exists(paths[key]):
            return False, f"MISSING_{key.upper()}"
    try:
        with open(paths["manifest"]) as f:
            bmanifest = json.load(f)
    except Exception:
        return False, "UNREADABLE_MANIFEST"
    if bmanifest.get("model", {}).get("revision") != MODEL_REVISION:
        return False, "MODEL_REVISION_MISMATCH"
    if bmanifest.get("dimension") != DIMENSION or bmanifest.get("dtype") != "float32":
        return False, "DIMENSION_OR_DTYPE_MISMATCH"
    expected_row_count = local_end - local_start
    if bmanifest.get("row_count") != expected_row_count:
        return False, "ROW_COUNT_MISMATCH"
    if bmanifest.get("global_start_index") != rows[local_start]["global_eligible_index"] or bmanifest.get("global_end_index") != rows[local_end - 1]["global_eligible_index"]:
        return False, "GLOBAL_RANGE_MISMATCH"
    actual_vectors_sha = sha256_file(paths["vectors"])
    if actual_vectors_sha != bmanifest.get("vectors_sha256"):
        return False, "VECTORS_SHA_MISMATCH"
    actual_mapping_sha = sha256_file(paths["mapping"])
    if actual_mapping_sha != bmanifest.get("mapping_sha256"):
        return False, "MAPPING_SHA_MISMATCH"
    arr = read_npy_float32_matrix(paths["vectors"])
    if arr.shape != (expected_row_count, DIMENSION):
        return False, "VECTOR_SHAPE_MISMATCH"
    if not np.isfinite(arr).all():
        return False, "NON_FINITE_VALUES"
    norms = np.linalg.norm(arr, axis=1)
    if np.max(np.abs(norms - 1.0)) > 0.01:
        return False, "NORMALIZATION_MISMATCH"
    return True, "OK"

resume_report = []
blocks_to_run = []
for block_idx, (local_start, local_end) in enumerate(blocks):
    valid, reason = block_is_valid(block_idx, local_start, local_end)
    resume_report.append({"block_idx": block_idx, "valid_reused": valid, "reason": reason})
    if not valid:
        blocks_to_run.append(block_idx)

print(json.dumps({"total_blocks": len(blocks), "blocks_to_run": blocks_to_run, "resume_report": resume_report}, indent=2))

In [ ]:
"""Cell 5 -- run every block that is not already validly checkpointed.
Bounded batch size, fixed deterministic row order within the block. On
CUDA OOM: halve batch_size and restart the WHOLE current block from
scratch -- never skip/drop individual rows, never write a partial block.
NaN/Inf in any output vector is rejected outright (not clamped, not
silently accepted). Vectors are L2-normalized by sentence-transformers'
own normalize_embeddings=True; re-verified explicitly below regardless."""
def embed_block(block_rows, batch_size):
    texts = [r["text"] for r in block_rows]
    started = time.perf_counter()
    vectors = model.encode(
        texts, batch_size=batch_size, normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=False,
    ).astype(np.float32)
    elapsed_ms = (time.perf_counter() - started) * 1000
    return vectors, elapsed_ms

for block_idx in blocks_to_run:
    local_start, local_end = blocks[block_idx]
    block_rows = rows[local_start:local_end]
    paths = block_paths(block_idx)
    # Clear out any stale/partial artifacts from a prior failed attempt at
    # this SAME block before writing new ones -- a half-written block must
    # never be mistaken for a valid one (block_is_valid already guards
    # this on the READ side; this is the WRITE-side counterpart).
    for key in ("vectors", "mapping", "manifest", "marker"):
        if os.path.exists(paths[key]):
            os.remove(paths[key])

    batch_size = BATCH_SIZE_INITIAL
    vectors = None
    elapsed_ms = None
    attempt = 0
    while vectors is None:
        attempt += 1
        try:
            vectors, elapsed_ms = embed_block(block_rows, batch_size)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if batch_size <= 1:
                raise RuntimeError(f"OOM_AT_BATCH_SIZE_1: block {block_idx} cannot complete on this GPU")
            batch_size = max(1, batch_size // 2)
            print(f"[full-shard-colab] block {block_idx}: CUDA OOM on attempt {attempt} -- halving batch_size to {batch_size} and restarting this BLOCK from scratch")

    if not np.isfinite(vectors).all():
        raise RuntimeError(f"NON_FINITE_OUTPUT: block {block_idx} produced NaN/Inf -- refusing to checkpoint")
    norms = np.linalg.norm(vectors, axis=1)
    if np.max(np.abs(norms - 1.0)) > 0.01:
        raise RuntimeError(f"NORMALIZATION_FAILURE: block {block_idx} worst |norm-1|={np.max(np.abs(norms - 1.0))}")

    write_npy_float32_matrix(paths["vectors"], vectors)
    with open(paths["mapping"], "w") as f:
        for r in block_rows:
            f.write(json.dumps({"global_eligible_index": r["global_eligible_index"], "embedding_input_id": r["embedding_input_id"], "embed_text_sha256": r["embed_text_sha256"]}) + "\n")

    block_manifest = {
        "schema_version": "p11f0-colab-full-shard-block-manifest.v1",
        "shard_id": SHARD_ID,
        "block_idx": block_idx,
        "global_start_index": block_rows[0]["global_eligible_index"],
        "global_end_index": block_rows[-1]["global_eligible_index"],
        "row_count": len(block_rows),
        "model": {"repository": MODEL_REPOSITORY, "revision": MODEL_REVISION},
        "dimension": DIMENSION,
        "dtype": "float32",
        "batch_size": batch_size,
        "elapsed_ms": elapsed_ms,
        "device": GPU_NAME,
        "runtime_versions": runtime_versions,
        "vectors_sha256": sha256_file(paths["vectors"]),
        "mapping_sha256": sha256_file(paths["mapping"]),
        "generated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    with open(paths["manifest"], "w") as f:
        json.dump(block_manifest, f)
    # Completion marker written LAST and atomically (write-then-rename) --
    # its mere existence is never trusted alone (block_is_valid always
    # re-verifies the SHA-256/shape/finite/norm facts too), but a marker
    # written before the manifest/vectors/mapping would be a false signal.
    tmp_marker = paths["marker"] + ".partial"
    with open(tmp_marker, "w") as f:
        f.write("complete\n")
    os.replace(tmp_marker, paths["marker"])
    print(f"[full-shard-colab] block {block_idx} done: rows={len(block_rows)} batch_size={batch_size} elapsed_ms={elapsed_ms:.0f}")

print(f"[full-shard-colab] all {len(blocks)} blocks present and valid for shard {SHARD_ID}")

In [ ]:
"""Cell 6 -- shard finalize. Verifies full block coverage (every row
covered exactly once, in order, no gap/overlap), then either merges all
blocks into a single per-shard vectors/mapping file (when that does not
require an unnecessary full duplicate copy sitting alongside the blocks)
or keeps the block set itself as the authoritative result -- the shard
manifest records EXACTLY which choice was made, so a downstream reader
never has to guess."""
covered_global_indices = []
for block_idx, (local_start, local_end) in enumerate(blocks):
    valid, reason = block_is_valid(block_idx, local_start, local_end)
    if not valid:
        raise RuntimeError(f"SHARD_FINALIZE_BLOCKED: block {block_idx} is not validly checkpointed ({reason}) -- re-run Cell 4/5")
    covered_global_indices.extend(range(rows[local_start]["global_eligible_index"], rows[local_end - 1]["global_eligible_index"] + 1))

expected_full_range = list(range(shard_meta["global_start_index"], shard_meta["global_end_index"] + 1))
if covered_global_indices != expected_full_range:
    raise RuntimeError("SHARD_FINALIZE_BLOCKED: block coverage does not exactly equal the shard's full contiguous global range (gap, overlap, or out-of-order)")

FINALIZE_AS_SINGLE_FILE = True  # flip to False to always keep the block set authoritative

result_manifest = {
    "schema_version": "p11f0-colab-full-shard-result-manifest.v1",
    "shard_id": SHARD_ID,
    "shard_gz_sha256": shard_meta["compressed_file_sha256"],
    "row_count": len(rows),
    "global_start_index": shard_meta["global_start_index"],
    "global_end_index": shard_meta["global_end_index"],
    "block_size": BLOCK_SIZE,
    "block_count": len(blocks),
    "model": {"repository": MODEL_REPOSITORY, "revision": MODEL_REVISION, "dimension": DIMENSION, "dtype": "float32"},
    "runner": RUNNER,
    "device": GPU_NAME,
    "runtime_versions": runtime_versions,
    "generated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}

if FINALIZE_AS_SINGLE_FILE:
    merged = np.concatenate([read_npy_float32_matrix(block_paths(i)["vectors"]) for i in range(len(blocks))], axis=0)
    merged_path = f"{SHARD_OUT_DIR}/shard-{shard_str}-vectors.npy"
    write_npy_float32_matrix(merged_path, merged)
    merged_mapping_path = f"{SHARD_OUT_DIR}/shard-{shard_str}-row-mapping.jsonl"
    with open(merged_mapping_path, "w") as f:
        for r in rows:
            f.write(json.dumps({"global_eligible_index": r["global_eligible_index"], "embedding_input_id": r["embedding_input_id"], "embed_text_sha256": r["embed_text_sha256"]}) + "\n")
    result_manifest["finalize_mode"] = "SINGLE_FILE"
    result_manifest["single_file_vectors_sha256"] = sha256_file(merged_path)
    result_manifest["single_file_mapping_sha256"] = sha256_file(merged_mapping_path)
    # The block set is left on disk too (not deleted) -- it remains a valid,
    # independently re-verifiable resume/audit trail even after finalize.
else:
    result_manifest["finalize_mode"] = "BLOCK_SET_AUTHORITATIVE"

result_manifest_path = f"{SHARD_OUT_DIR}/shard-{shard_str}-result-manifest.json"
with open(result_manifest_path, "w") as f:
    json.dump(result_manifest, f, indent=2)

print(json.dumps(result_manifest, indent=2))